# ForestWatch AI — satellite pipeline

Neural Storm 2026 · GP-04 · Working notebook

This notebook verifies Earth Engine access and checks Sentinel-2 data for a pilot area. A satellite-observed change is only a candidate for human review; it is not proof of illegal deforestation.

**Run the two code cells from top to bottom. Do not skip the setup cell.**

## 1. Install required packages

Run this cell first. It installs the Earth Engine Python API and scikit-learn for the unsupervised model.

In [ ]:
%pip -q install earthengine-api scikit-learn

## 2. Authenticate, initialize and check satellite data

Run this entire cell after Step 1. If Google asks you to authorize Earth Engine, follow the on-screen prompts.

In [ ]:
import ee

PROJECT_ID = 'forestwatch-ai-2026'
START_DATE = '2025-01-01'
END_DATE = '2025-12-31'

print('Authenticating with Google Earth Engine...')
ee.Authenticate()
ee.Initialize(project=PROJECT_ID)
print('Earth Engine initialized successfully.')

# Pilot test area in the central Western Ghats.
# This rectangle is only a data-access test area, not a final study boundary.
aoi = ee.Geometry.Rectangle([74.5, 12.0, 75.5, 14.0])

sentinel2 = (
    ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
    .filterBounds(aoi)
    .filterDate(START_DATE, END_DATE)
)

scene_count = sentinel2.size().getInfo()
print('\n--- ForestWatch data access check ---')
print(f'Earth Engine project: {PROJECT_ID}')
print(f'Date window: {START_DATE} to {END_DATE}')
print(f'Sentinel-2 scenes found: {scene_count}')
print('Result: Sentinel-2 collection is accessible.' if scene_count > 0 else 'Result: No scenes found for this area and date window.')

## 3. First vegetation-change screening (pilot)

This cell builds cloud-screened Sentinel-2 NDVI composites for the same Jan–Mar season in 2024 and 2025, then maps pixels with a large NDVI decrease. This is an exploratory vegetation-change screen only—not a deforestation classifier or proof of illegal activity. The pilot box is deliberately smaller than the initial data-access rectangle to keep processing manageable.

In [ ]:
# Re-initialize safely if this cell is run after a Colab runtime reset.
import ee
from IPython.display import display, Image

PROJECT_ID = 'forestwatch-ai-2026'
try:
    ee.Initialize(project=PROJECT_ID)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=PROJECT_ID)

# Small pilot area in the Western Ghats; not a final forest boundary.
pilot_aoi = ee.Geometry.Rectangle([74.80, 13.00, 75.00, 13.20])

def mask_s2_clouds(image):
    scl = image.select('SCL')
    # Remove cloud shadow, medium/high cloud, cirrus and snow classes.
    clear = (scl.neq(3)
             .And(scl.neq(8))
             .And(scl.neq(9))
             .And(scl.neq(10))
             .And(scl.neq(11)))
    return image.updateMask(clear).select(['B4', 'B8']).copyProperties(image, ['system:time_start'])

def make_seasonal_ndvi(start_date, end_date):
    collection = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
                  .filterBounds(pilot_aoi)
                  .filterDate(start_date, end_date)
                  .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 80))
                  .map(mask_s2_clouds))
    count = collection.size().getInfo()
    if count == 0:
        raise RuntimeError(f'No Sentinel-2 scenes found for {start_date} to {end_date}.')
    composite = collection.median()
    ndvi = composite.normalizedDifference(['B8', 'B4']).rename('NDVI')
    return ndvi, count

print('Building season-matched NDVI composites...')
ndvi_2024, count_2024 = make_seasonal_ndvi('2024-01-01', '2024-04-01')
ndvi_2025, count_2025 = make_seasonal_ndvi('2025-01-01', '2025-04-01')
ndvi_change = ndvi_2025.subtract(ndvi_2024).rename('NDVI_change').clip(pilot_aoi)

# Conservative exploratory screen: previously greener pixels with a large decline.
# Threshold is a screening parameter, not a validated legal/deforestation rule.
candidate_pixels = ndvi_2024.gt(0.5).And(ndvi_change.lt(-0.20)).rename('candidate')
candidate_area_ha = (ee.Image.pixelArea().divide(10000)
                     .updateMask(candidate_pixels)
                     .rename('candidate_area_ha'))

mean_values = ee.Image.cat([ndvi_2024.rename('mean_NDVI_2024'),
                            ndvi_2025.rename('mean_NDVI_2025'),
                            ndvi_change.rename('mean_NDVI_change')]).reduceRegion(
    reducer=ee.Reducer.mean(), geometry=pilot_aoi, scale=100,
    maxPixels=1e8, tileScale=4).getInfo()

area_values = candidate_area_ha.reduceRegion(
    reducer=ee.Reducer.sum(), geometry=pilot_aoi, scale=100,
    maxPixels=1e8, tileScale=4).getInfo()

print('\n--- ForestWatch pilot screening summary ---')
print(f'Sentinel-2 scenes used, Jan–Mar 2024: {count_2024}')
print(f'Sentinel-2 scenes used, Jan–Mar 2025: {count_2025}')
print('Mean NDVI and mean change:', mean_values)
print('Exploratory candidate area (hectares):', area_values.get('candidate_area_ha', 0) or 0)
print('Interpretation: investigate candidate pixels; do not label them illegal deforestation.')

thumbnail_url = ndvi_change.getThumbURL({
    'region': pilot_aoi, 'dimensions': 700, 'format': 'png',
    'min': -0.4, 'max': 0.4,
    'palette': ['8c2d04', 'd8b365', 'f5f5f5', '5ab4ac', '01665e']
})
display(Image(url=thumbnail_url))

## Interpretation and limitations

- A negative NDVI change can result from harvest, fire, drought, seasonal variation, cloud contamination or other causes.
- The Jan–Mar windows are season-matched but do not eliminate all weather or phenology differences.
- The 0.5 baseline and −0.20 change thresholds are exploratory settings and must be tested against reference labels before any performance claim.
- The reported area is a coarse screening estimate at 100 m reduction scale, not a survey-grade boundary.
- Next we will add data-quality checks, alternative explanations and independent reference validation before calling any area a high-priority case.

## What happens next?

Once this access check succeeds, we will add cloud and shadow screening, season-matched composites, NDVI summaries and a transparent change-evidence table. We will validate each step before adding it to the dashboard.

## 4. Turn noisy changed pixels into candidate regions

This step groups adjacent candidate pixels into connected regions, removes very small clusters, and ranks the remaining regions by mapped area. It uses the variables created by Section 3, so run the notebook from the top in order. The minimum cluster size is a technical noise filter, not a scientific or legal threshold.

In [ ]:
import ee
from IPython.display import display, Image

# Check that the previous satellite-analysis section has been run.
required_names = ['pilot_aoi', 'ndvi_change', 'candidate_pixels']
missing_names = [name for name in required_names if name not in globals()]
if missing_names:
    raise RuntimeError('Please run Sections 1–3 from the top first. Missing: ' + ', '.join(missing_names))

# Count neighbouring candidate pixels. Eight-way connectivity includes diagonals.
connected_count = candidate_pixels.selfMask().connectedPixelCount(
    maxSize=100, eightConnected=True
)

# Keep connected groups containing at least 8 native-resolution pixels.
# At Sentinel-2 10 m resolution, 8 pixels represent about 0.08 ha before geometry effects.
clean_candidate_mask = (candidate_pixels
                        .updateMask(connected_count.gte(8))
                        .selfMask()
                        .toInt()
                        .rename('cluster'))

# Vectorize connected groups and summarize the NDVI change within each group.
vector_input = clean_candidate_mask.addBands(ndvi_change.rename('ndvi_change'))
candidate_regions = vector_input.reduceToVectors(
    reducer=ee.Reducer.mean(),
    geometry=pilot_aoi,
    scale=10,
    geometryType='polygon',
    eightConnected=True,
    labelProperty='cluster_id',
    maxPixels=100000000,
    tileScale=4
)

def add_region_metrics(feature):
    area_ha = feature.geometry().area(maxError=1).divide(10000)
    center = feature.geometry().centroid(maxError=1).coordinates()
    return feature.set({
        'area_ha': area_ha,
        'longitude': center.get(0),
        'latitude': center.get(1),
        'mean_ndvi_change': feature.get('mean')
    })

regions_with_metrics = candidate_regions.map(add_region_metrics)
# Exclude mapped fragments smaller than 0.1 ha, then show the 20 largest.
review_regions = (regions_with_metrics
                  .filter(ee.Filter.gte('area_ha', 0.1))
                  .sort('area_ha', False))

region_count = review_regions.size().getInfo()
top_regions = review_regions.limit(20).getInfo()

print('--- ForestWatch candidate-region summary ---')
print('Connected candidate regions >= 0.1 ha:', region_count)
print('Showing up to 20 largest regions:')
for index, feature in enumerate(top_regions.get('features', []), start=1):
    props = feature.get('properties', {})
    print(
        f"{index}. Area: {props.get('area_ha', float('nan')):.2f} ha | "
        f"Centre: {props.get('latitude', float('nan')):.5f}, "
        f"{props.get('longitude', float('nan')):.5f} | "
        f"Mean NDVI change: {props.get('mean_ndvi_change', float('nan')):.3f}"
    )

# Render the grouped regions as a simple image for visual inspection.
region_outline = ee.Image().byte().paint(
    featureCollection=review_regions,
    color=1,
    width=2
).clip(pilot_aoi)
region_thumb = region_outline.getThumbURL({
    'region': pilot_aoi,
    'dimensions': 700,
    'format': 'png',
    'min': 0,
    'max': 1,
    'palette': ['ffffff', 'b2182b']
})
display(Image(url=region_thumb))

print('\nReminder: these are vegetation-change candidates for human review, not confirmed deforestation.')

## Candidate-region interpretation

Connected-region grouping reduces isolated speckle but does not establish the cause of a change. Region area and centroid are approximate map summaries. Before assigning investigation priority, we must inspect image quality, compare radar evidence, check fire and other disturbance sources where possible, and validate against independent reference data.

## 5. View candidate regions over real satellite imagery

The previous red-on-white image was an outline-only diagnostic layer. This section places those outlines over a true-colour Sentinel-2 composite so the candidate shapes can be inspected in geographic context.

In [ ]:
import ee
from IPython.display import display, Image

required_names = ['pilot_aoi', 'review_regions']
missing_names = [name for name in required_names if name not in globals()]
if missing_names:
    raise RuntimeError('Please run Sections 1–4 from the top first. Missing: ' + ', '.join(missing_names))

def mask_s2_rgb(image):
    scl = image.select('SCL')
    clear = (scl.neq(3)
             .And(scl.neq(8))
             .And(scl.neq(9))
             .And(scl.neq(10))
             .And(scl.neq(11)))
    return image.updateMask(clear).select(['B4', 'B3', 'B2'])

rgb_collection = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
                  .filterBounds(pilot_aoi)
                  .filterDate('2025-01-01', '2025-04-01')
                  .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 80))
                  .map(mask_s2_rgb))

rgb_count = rgb_collection.size().getInfo()
if rgb_count == 0:
    raise RuntimeError('No Sentinel-2 RGB scenes are available for the 2025 pilot period.')

rgb_composite = rgb_collection.median().clip(pilot_aoi)
rgb_visual = rgb_composite.visualize(
    bands=['B4', 'B3', 'B2'], min=0, max=3000, gamma=1.2
)

outline = ee.Image().byte().paint(
    featureCollection=review_regions,
    color=1,
    width=2
).selfMask().clip(pilot_aoi)
outline_visual = outline.visualize(palette=['ff2d2d'], opacity=1)
overlay = rgb_visual.blend(outline_visual)

overlay_url = overlay.getThumbURL({
    'region': pilot_aoi,
    'dimensions': 900,
    'format': 'png'
})

print(f'Sentinel-2 scenes used for background composite: {rgb_count}')
print('Red outlines = candidate vegetation-change regions; inspect them against the underlying imagery.')
display(Image(url=overlay_url))

## 6. Add Sentinel-1 radar evidence

This section first diagnoses Sentinel-1 availability for the pilot area and both seasonal windows. It then uses a polarization and orbit direction actually present in both windows. If there is no common orbit direction, it stops rather than silently mixing acquisition geometries. Radar change is supporting evidence only, not proof of deforestation.

Run Sections 1–5 first in the same runtime.

In [ ]:
# Sentinel-1 availability diagnosis and season-matched radar comparison.
# Run Sections 1–5 first in this same Colab runtime.

import ee
from IPython.display import Image, display

required_names = ["pilot_aoi", "candidate_pixels", "ndvi_change"]
missing_names = [name for name in required_names if name not in globals()]
if missing_names:
    raise RuntimeError(
        "Run Sections 1–5 from the top first. Missing variables: "
        + ", ".join(missing_names)
    )

def s1_base(start_date, end_date):
    return (
        ee.ImageCollection("COPERNICUS/S1_GRD")
        .filterBounds(pilot_aoi)
        .filterDate(start_date, end_date)
        .filter(ee.Filter.eq("instrumentMode", "IW"))
    )

periods = {
    "2024": ("2024-01-01", "2024-04-01"),
    "2025": ("2025-01-01", "2025-04-01")
}

collections = {}
print("--- Sentinel-1 availability diagnosis ---")
for year, dates in periods.items():
    base = s1_base(*dates)
    vv = base.filter(
        ee.Filter.listContains("transmitterReceiverPolarisation", "VV")
    )
    vv_vh = vv.filter(
        ee.Filter.listContains("transmitterReceiverPolarisation", "VH")
    )
    ascending = vv_vh.filter(
        ee.Filter.eq("orbitProperties_pass", "ASCENDING")
    )
    descending = vv_vh.filter(
        ee.Filter.eq("orbitProperties_pass", "DESCENDING")
    )

    counts = {
        "IW scenes": base.size().getInfo(),
        "IW with VV": vv.size().getInfo(),
        "IW with VV+VH": vv_vh.size().getInfo(),
        "VV+VH ascending": ascending.size().getInfo(),
        "VV+VH descending": descending.size().getInfo()
    }
    print(year, counts)
    collections[year] = {
        "vv": vv,
        "vv_vh": vv_vh,
        "ascending": ascending,
        "descending": descending
    }

# Prefer a polarization available in both periods and a common orbit direction.
use_vh = (
    collections["2024"]["vv_vh"].size().getInfo() > 0
    and collections["2025"]["vv_vh"].size().getInfo() > 0
)
polarization = ["VV", "VH"] if use_vh else ["VV"]

common_passes = []
collection_key = "vv_vh" if use_vh else "vv"
for orbit_pass in ["ASCENDING", "DESCENDING"]:
    count_2024 = collections["2024"][collection_key].filter(
        ee.Filter.eq("orbitProperties_pass", orbit_pass)
    ).size().getInfo()
    count_2025 = collections["2025"][collection_key].filter(
        ee.Filter.eq("orbitProperties_pass", orbit_pass)
    ).size().getInfo()
    if count_2024 > 0 and count_2025 > 0:
        common_passes.append((min(count_2024, count_2025), orbit_pass))

if not common_passes:
    raise RuntimeError(
        "Sentinel-1 scenes were found, but no common orbit direction is "
        "available in both seasonal windows. Radar comparison has stopped "
        "to avoid mixing acquisition geometries. Use the counts above to "
        "choose a different matched date window."
    )

common_passes.sort(reverse=True)
selected_pass = common_passes[0][1]
print("\nSelected polarization:", " + ".join(polarization))
print("Selected common orbit direction:", selected_pass)

s1_2024 = (
    collections["2024"][collection_key]
    .filter(ee.Filter.eq("orbitProperties_pass", selected_pass))
    .select(polarization)
)
s1_2025 = (
    collections["2025"][collection_key]
    .filter(ee.Filter.eq("orbitProperties_pass", selected_pass))
    .select(polarization)
)

count_s1_2024 = s1_2024.size().getInfo()
count_s1_2025 = s1_2025.size().getInfo()
print("Scenes used in 2024:", count_s1_2024)
print("Scenes used in 2025:", count_s1_2025)

radar_2024 = s1_2024.median().clip(pilot_aoi)
radar_2025 = s1_2025.median().clip(pilot_aoi)

vv_change = (
    radar_2025.select("VV")
    .subtract(radar_2024.select("VV"))
    .rename("VV_change_dB")
)

change_bands = [vv_change]
if use_vh:
    vh_change = (
        radar_2025.select("VH")
        .subtract(radar_2024.select("VH"))
        .rename("VH_change_dB")
    )
    change_bands.append(vh_change)

# This is only a transparent exploratory flag, not a validated threshold.
# If VH is unavailable, do not manufacture a VH-based support layer.
if use_vh:
    radar_support = vh_change.lt(-1.0).rename("radar_support")
    optical_and_radar = (
        candidate_pixels.eq(1)
        .And(radar_support)
        .rename("optical_radar_overlap")
    )
    change_bands.append(optical_and_radar)

summary = ee.Image.cat(change_bands).reduceRegion(
    reducer=ee.Reducer.mean(),
    geometry=pilot_aoi,
    scale=100,
    maxPixels=100000000,
    tileScale=4
).getInfo()

print("\n--- Sentinel-1 supporting-evidence summary ---")
print("Mean VV change (dB):", summary.get("VV_change_dB"))
if use_vh:
    print("Mean VH change (dB):", summary.get("VH_change_dB"))
    print(
        "Mean optical/radar overlap fraction:",
        summary.get("optical_radar_overlap")
    )
else:
    print("VH unavailable in one or both windows; no VV/VH overlap score was calculated.")

print(
    "Interpretation: radar change is supporting evidence only. "
    "It does not establish deforestation or illegality."
)

radar_band_to_display = vh_change if use_vh else vv_change
radar_thumbnail_url = radar_band_to_display.getThumbURL({
    "region": pilot_aoi,
    "dimensions": 800,
    "format": "png",
    "min": -3,
    "max": 3,
    "palette": ["8c2d04", "f7f7f7", "2166ac"]
})
display(Image(url=radar_thumbnail_url))


## Sentinel-1 interpretation and safeguards

- Radar backscatter changes are not uniquely associated with forest clearing.
- Soil moisture, rainfall, vegetation structure, slope/aspect and acquisition geometry can change VV/VH values.
- The −1 dB VH threshold is an exploratory screening parameter, not a literature-validated decision boundary for this site.
- Do not call optical/radar overlap a confirmed deforestation event. Inspect dates, imagery, topography, fire/landslide context and independent reference labels.
- Before reporting performance, create a spatially independent labelled validation sample and report precision, recall and uncertainty.

## 7. Add per-pixel optical observation quality control

This step rebuilds the 2024 and 2025 seasonal NDVI composites while counting valid, cloud-screened observations for every pixel. Pixels with fewer than three valid observations in either period are excluded from the updated change screen. The threshold is a transparent quality-control setting, not a validated deforestation threshold.

This cell updates `ndvi_2024`, `ndvi_2025`, `ndvi_change`, and `candidate_pixels`. After it succeeds, rerun Sections 4, 5 and 6 so the regions, map and radar comparison use the quality-screened candidates.

In [ ]:
# Rebuild the optical screen with stricter scene-classification masking
# and a per-pixel minimum number of clear observations.

import ee
from IPython.display import Image, display

if "pilot_aoi" not in globals():
    raise RuntimeError("Run Sections 1–3 first so pilot_aoi is available.")

MIN_VALID_OBSERVATIONS = 3

def make_quality_ndvi_collection(start_date, end_date):
    collection = (
        ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
        .filterBounds(pilot_aoi)
        .filterDate(start_date, end_date)
        .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 80))
    )

    def to_masked_ndvi(image):
        scl = image.select("SCL")
        # Exclude saturated/defective, cloud shadow, unclassified,
        # medium/high cloud, cirrus and snow pixels.
        clear = (
            scl.neq(1)
            .And(scl.neq(3))
            .And(scl.neq(7))
            .And(scl.neq(8))
            .And(scl.neq(9))
            .And(scl.neq(10))
            .And(scl.neq(11))
        )
        masked = image.updateMask(clear)
        ndvi = masked.normalizedDifference(["B8", "B4"]).rename("NDVI")
        return ndvi.copyProperties(image, ["system:time_start"])

    return collection.map(to_masked_ndvi)

print("--- Building quality-screened optical composites ---")
ndvi_collection_2024 = make_quality_ndvi_collection(
    "2024-01-01", "2024-04-01"
)
ndvi_collection_2025 = make_quality_ndvi_collection(
    "2025-01-01", "2025-04-01"
)

scene_count_2024 = ndvi_collection_2024.size().getInfo()
scene_count_2025 = ndvi_collection_2025.size().getInfo()

if scene_count_2024 == 0 or scene_count_2025 == 0:
    raise RuntimeError(
        "No Sentinel-2 scenes are available in one of the seasonal windows."
    )

valid_obs_2024 = ndvi_collection_2024.count().rename("valid_obs_2024")
valid_obs_2025 = ndvi_collection_2025.count().rename("valid_obs_2025")

ndvi_2024 = (
    ndvi_collection_2024.median()
    .updateMask(valid_obs_2024.gte(MIN_VALID_OBSERVATIONS))
    .clip(pilot_aoi)
)
ndvi_2025 = (
    ndvi_collection_2025.median()
    .updateMask(valid_obs_2025.gte(MIN_VALID_OBSERVATIONS))
    .clip(pilot_aoi)
)

# Replace the earlier exploratory change layer with quality-screened data.
ndvi_change = ndvi_2025.subtract(ndvi_2024).rename("NDVI_change").clip(pilot_aoi)

# Retain the same exploratory vegetation-change thresholds for comparison.
candidate_pixels = (
    ndvi_2024.gt(0.5)
    .And(ndvi_change.lt(-0.20))
    .rename("candidate")
)

candidate_area_ha = (
    ee.Image.pixelArea()
    .divide(10000)
    .updateMask(candidate_pixels)
    .rename("candidate_area_ha")
)

valid_stats_2024 = valid_obs_2024.reduceRegion(
    reducer=ee.Reducer.mean(),
    geometry=pilot_aoi,
    scale=20,
    maxPixels=100000000,
    tileScale=4
).getInfo()

valid_stats_2025 = valid_obs_2025.reduceRegion(
    reducer=ee.Reducer.mean(),
    geometry=pilot_aoi,
    scale=20,
    maxPixels=100000000,
    tileScale=4
).getInfo()

candidate_pixel_count = candidate_pixels.selfMask().reduceRegion(
    reducer=ee.Reducer.count(),
    geometry=pilot_aoi,
    scale=10,
    maxPixels=100000000,
    tileScale=4
).getInfo()

candidate_area = candidate_area_ha.reduceRegion(
    reducer=ee.Reducer.sum(),
    geometry=pilot_aoi,
    scale=10,
    maxPixels=100000000,
    tileScale=4
).getInfo()

print("\n--- Optical quality-control summary ---")
print("Sentinel-2 scenes in 2024:", scene_count_2024)
print("Sentinel-2 scenes in 2025:", scene_count_2025)
print("Mean valid observations per pixel, 2024:", valid_stats_2024)
print("Mean valid observations per pixel, 2025:", valid_stats_2025)
print("Candidate pixels after observation-quality mask:", candidate_pixel_count)
print("Candidate area (hectares, approximate):", candidate_area.get("candidate_area_ha", 0) or 0)
print(
    "Quality rule: pixels need at least",
    MIN_VALID_OBSERVATIONS,
    "clear observations in each period."
)
print("These remain vegetation-change candidates, not confirmed deforestation.")

quality_map = ndvi_change.getThumbURL({
    "region": pilot_aoi,
    "dimensions": 800,
    "format": "png",
    "min": -0.4,
    "max": 0.4,
    "palette": ["8c2d04", "d8b365", "f5f5f5", "5ab4ac", "01665e"]
})
display(Image(url=quality_map))


## 8. Screen for persistent vegetation decline (2023–2025)

This section compares the same Jan–Mar season across three years. A pixel is retained only when it has at least the Section 7 minimum number of clear observations in every year and its 2024 and 2025 seasonal NDVI are both at least 0.15 below the 2023 baseline. This is a transparent exploratory persistence rule, not a validated deforestation threshold. It can miss recent disturbances and can still include non-forest land-cover changes.

Run Section 8 after Section 7. If it succeeds, rerun Sections 4, 5 and 6 so the region map and radar comparison use the persistent candidate layer.

In [ ]:
# Three-year season-matched persistence screen (2023–2025).
# Requires Sections 1–3 and 7 to have run in this Colab runtime.

import ee
from IPython.display import Image, display

if "pilot_aoi" not in globals():
    raise RuntimeError("Run Sections 1–3 first so pilot_aoi is available.")
if "MIN_VALID_OBSERVATIONS" not in globals():
    raise RuntimeError("Run Section 7 first so the observation-quality rule is available.")

def make_persistence_collection(start_date, end_date):
    collection = (
        ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
        .filterBounds(pilot_aoi)
        .filterDate(start_date, end_date)
        .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 80))
    )

    def to_masked_ndvi(image):
        scl = image.select("SCL")
        clear = (
            scl.neq(1)
            .And(scl.neq(3))
            .And(scl.neq(7))
            .And(scl.neq(8))
            .And(scl.neq(9))
            .And(scl.neq(10))
            .And(scl.neq(11))
        )
        ndvi = image.updateMask(clear).normalizedDifference(["B8", "B4"])
        return ndvi.rename("NDVI").copyProperties(image, ["system:time_start"])

    return collection.map(to_masked_ndvi)

periods = {
    "2023": ("2023-01-01", "2023-04-01"),
    "2024": ("2024-01-01", "2024-04-01"),
    "2025": ("2025-01-01", "2025-04-01")
}

collections = {
    year: make_persistence_collection(*dates)
    for year, dates in periods.items()
}

scene_counts = {
    year: collection.size().getInfo()
    for year, collection in collections.items()
}
print("--- Three-year optical persistence screening ---")
print("Sentinel-2 scenes by year:", scene_counts)

if any(count == 0 for count in scene_counts.values()):
    raise RuntimeError(
        "At least one Jan–Mar seasonal window has no Sentinel-2 scenes. "
        "No persistence result was calculated."
    )

valid_counts = {
    year: collection.count().rename("valid_obs_" + year)
    for year, collection in collections.items()
}

seasonal_ndvi = {
    year: collection.median()
        .updateMask(valid_counts[year].gte(MIN_VALID_OBSERVATIONS))
        .clip(pilot_aoi)
    for year, collection in collections.items()
}

# Require adequate clear observations in all three years.
valid_all_years = (
    valid_counts["2023"].gte(MIN_VALID_OBSERVATIONS)
    .And(valid_counts["2024"].gte(MIN_VALID_OBSERVATIONS))
    .And(valid_counts["2025"].gte(MIN_VALID_OBSERVATIONS))
)

# Persistent lower vegetation signal: both later seasons remain at least
# 0.15 NDVI below the 2023 seasonal baseline. This is an exploratory rule.
drop_2024 = seasonal_ndvi["2023"].subtract(seasonal_ndvi["2024"])
drop_2025 = seasonal_ndvi["2023"].subtract(seasonal_ndvi["2025"])

persistent_candidates = (
    valid_all_years
    .And(drop_2024.gte(0.15))
    .And(drop_2025.gte(0.15))
    .rename("persistent_candidate")
    .selfMask()
    .clip(pilot_aoi)
)

# Preserve the previous two-year candidate layer for transparent comparison.
if "candidate_pixels" not in globals():
    raise RuntimeError("Section 7 must run before Section 8.")

persistent_area_image = (
    ee.Image.pixelArea().divide(10000)
    .updateMask(persistent_candidates)
    .rename("persistent_area_ha")
)

persistent_stats = ee.Image.cat([
    drop_2024.rename("baseline_drop_2024"),
    drop_2025.rename("baseline_drop_2025")
]).updateMask(persistent_candidates).reduceRegion(
    reducer=ee.Reducer.mean(),
    geometry=pilot_aoi,
    scale=20,
    maxPixels=100000000,
    tileScale=4
).getInfo()

persistent_pixel_count = persistent_candidates.reduceRegion(
    reducer=ee.Reducer.count(),
    geometry=pilot_aoi,
    scale=10,
    maxPixels=100000000,
    tileScale=4
).getInfo()

persistent_area = persistent_area_image.reduceRegion(
    reducer=ee.Reducer.sum(),
    geometry=pilot_aoi,
    scale=10,
    maxPixels=100000000,
    tileScale=4
).getInfo()

print("\n--- Persistence screening summary ---")
print("Mean baseline-to-2024 NDVI drop:", persistent_stats.get("baseline_drop_2024"))
print("Mean baseline-to-2025 NDVI drop:", persistent_stats.get("baseline_drop_2025"))
print("Persistent candidate pixels:", persistent_pixel_count.get("persistent_candidate", 0))
print("Persistent candidate area (ha, approximate):", persistent_area.get("persistent_area_ha", 0) or 0)
print("Persistence rule: NDVI is at least 0.15 below 2023 in both 2024 and 2025.")
print("This is an exploratory persistence screen, not proof of forest loss or illegality.")

# From this point onward, Sections 4–6 will use the stricter persistent layer.
candidate_pixels = persistent_candidates.rename("candidate")

persistence_map = ee.Image().byte().paint(
    ee.FeatureCollection([
        ee.Feature(pilot_aoi)
    ]),
    0
).visualize(min=0, max=1)

persistence_display = (
    ee.Image.constant(0).clip(pilot_aoi)
    .where(persistent_candidates, 1)
    .visualize(min=0, max=1, palette=["173f2a", "d73027"])
)

persistence_url = persistence_display.getThumbURL({
    "region": pilot_aoi,
    "dimensions": 800,
    "format": "png"
})
display(Image(url=persistence_url))


## 9. Build a region-level evidence table

This section summarizes optical persistence and radar support for each connected candidate region. Radar support is calculated only among pixels with valid radar observations; radar coverage is the separate fraction of all region pixels with observations. Regions are sorted by area for review; the measurements are evidence descriptors, not a legal or confirmed-deforestation classification.

Run after Sections 4–8.

In [ ]:
# Region-level evidence summary: optical persistence + radar support.
# Requires Sections 4, 6, 7 and 8 to have run successfully.

import ee

required_names = [
    "pilot_aoi", "review_regions", "drop_2024", "drop_2025",
    "ndvi_change", "vv_change", "vh_change", "radar_support"
]
missing_names = [name for name in required_names if name not in globals()]
if missing_names:
    raise RuntimeError(
        "Run Sections 4, 6, 7 and 8 first. Missing variables: "
        + ", ".join(missing_names)
    )

# A value of 1 means the pixel met the exploratory VH-change support rule.
# Keep support masked to observed radar pixels; report coverage separately.
radar_valid = vh_change.mask().unmask(0).rename("radar_valid_fraction")
radar_support_fraction = radar_support.rename("radar_support_fraction")

evidence_stack = ee.Image.cat([
    drop_2024.rename("baseline_ndvi_drop_2024"),
    drop_2025.rename("baseline_ndvi_drop_2025"),
    ndvi_change.rename("ndvi_change_2025_vs_2024"),
    radar_support_fraction,
    radar_valid
])

region_evidence = evidence_stack.reduceRegions(
    collection=review_regions,
    reducer=ee.Reducer.mean(),
    scale=20,
    tileScale=4
)

# Keep the largest 20 for a compact, judge-readable review table.
top_evidence = region_evidence.sort("area_ha", False).limit(20).getInfo()

print("--- Region-level multi-sensor evidence (largest 20) ---")
print(
    "Rank | Area ha | NDVI drop vs 2023 (2024 / 2025) | "
    "NDVI change 2025 vs 2024 | Radar support share | Radar coverage share"
)

features = top_evidence.get("features", [])
for rank, feature in enumerate(features, start=1):
    p = feature.get("properties", {})
    def fmt(value):
        return "NA" if value is None else f"{value:.3f}"
    print(
        f"{rank:>2} | {p.get('area_ha', 0):.2f} | "
        f"{fmt(p.get('baseline_ndvi_drop_2024'))} / "
        f"{fmt(p.get('baseline_ndvi_drop_2025'))} | "
        f"{fmt(p.get('ndvi_change_2025_vs_2024'))} | "
        f"{fmt(p.get('radar_support_fraction'))} | "
        f"{fmt(p.get('radar_valid_fraction'))}"
    )

print("\nInterpretation:")
print("- Large baseline NDVI drops in both years indicate persistent optical decline.")
print("- A near-zero 2025-vs-2024 change can mean the decline persisted rather than newly worsened.")
print("- Radar support share is the fraction of radar-observed pixels meeting the exploratory VH < -1 dB change rule.")
print("- Radar coverage share is reported separately; low coverage weakens radar interpretation.")
print("- These values prioritize human review; they do not establish forest loss, cause, or illegality.")


## 10. Restrict persistent candidates to baseline tree-cover areas

This validation step intersects the 2023–2025 persistent optical-decline layer with ESA WorldCover 2021 class 10 (Tree cover). It reports how much of the candidate area falls inside areas mapped as tree cover in the 2021 reference layer. This reduces obvious non-tree-cover candidates, but WorldCover is a 2021 land-cover map and does not prove that a pixel was natural forest, nor that any later change was illegal.

Run after Sections 1–8. Section 10 does not overwrite the candidate layer used by earlier sections.

In [ ]:
# Baseline tree-cover reference screen using ESA WorldCover 2021.
# Requires the persistent_candidates layer from Section 8.

import ee
from IPython.display import Image, display

if "pilot_aoi" not in globals() or "persistent_candidates" not in globals():
    raise RuntimeError("Run Sections 1–3 and Section 8 first.")

worldcover = (
    ee.ImageCollection("ESA/WorldCover/v200")
    .first()
    .select("Map")
    .clip(pilot_aoi)
)

# ESA WorldCover class 10 = Tree cover.
tree_cover_2021 = worldcover.eq(10).rename("tree_cover_2021")

# Keep persistent vegetation-decline candidates that overlap the 2021
# tree-cover class. This is a reference-screened subset, not a forest-loss label.
tree_cover_persistent_candidates = (
    persistent_candidates
    .updateMask(tree_cover_2021)
    .rename("tree_cover_persistent_candidate")
    .clip(pilot_aoi)
)

area_ha = ee.Image.pixelArea().divide(10000).rename("area_ha")

def sum_area(image):
    result = area_ha.updateMask(image).reduceRegion(
        reducer=ee.Reducer.sum(),
        geometry=pilot_aoi,
        scale=10,
        maxPixels=100000000,
        tileScale=4
    ).get("area_ha")
    return ee.Number(ee.Algorithms.If(result, result, 0))

persistent_area_ha = sum_area(persistent_candidates)
tree_cover_area_ha = sum_area(tree_cover_2021)
tree_cover_candidate_area_ha = sum_area(tree_cover_persistent_candidates)

candidate_overlap_pct = ee.Number(
    ee.Algorithms.If(
        persistent_area_ha.gt(0),
        tree_cover_candidate_area_ha.divide(persistent_area_ha).multiply(100),
        0
    )
)

print("--- Baseline tree-cover reference screen ---")
print("ESA WorldCover reference year: 2021")
print("Reference class: 10 (Tree cover)")
print("Persistent candidate area (ha, approximate):", persistent_area_ha.getInfo())
print("2021 tree-cover area in pilot AOI (ha, approximate):", tree_cover_area_ha.getInfo())
print("Persistent candidates inside 2021 tree cover (ha, approximate):", tree_cover_candidate_area_ha.getInfo())
print("Share of persistent candidate area overlapping 2021 tree cover (%):", candidate_overlap_pct.getInfo())
print("Interpretation: this is a baseline land-cover cross-check, not confirmation of deforestation or illegality.")

# Display the reference-screened candidates on a dark background.
screened_display = (
    ee.Image.constant(0).clip(pilot_aoi)
    .where(tree_cover_persistent_candidates, 1)
    .visualize(min=0, max=1, palette=["183b2a", "ff4d4d"])
)
screened_url = screened_display.getThumbURL({
    "region": pilot_aoi,
    "dimensions": 800,
    "format": "png"
})
display(Image(url=screened_url))


## 11. Rank tree-cover-overlap candidate regions for review

This section groups Section 10 tree-cover-overlap candidates into connected regions, removes fragments below 0.1 ha, then measures optical and radar evidence separately for each region. Radar support is calculated among observed radar pixels; radar coverage is reported separately. This is a human-review shortlist, not a finding of illegal activity.

Run Sections 1–8 and 10 first in the current runtime.


In [ ]:
# Build a review shortlist and measure evidence in a separate regional pass.

import ee
from IPython.display import Image, display

required_names = [
    "pilot_aoi", "tree_cover_persistent_candidates",
    "drop_2024", "drop_2025", "ndvi_change", "vh_change", "radar_support"
]
missing_names = [name for name in required_names if name not in globals()]
if missing_names:
    raise RuntimeError(
        "Run Sections 1–8 and 10 first. Missing: " + ", ".join(missing_names)
    )

# Create connected regions from the candidate mask only.
review_mask = (
    tree_cover_persistent_candidates.selfMask().toInt()
    .rename("candidate_cluster")
)
connected = review_mask.connectedPixelCount(maxSize=256, eightConnected=True)
review_mask = review_mask.updateMask(connected.gte(8))

regions = review_mask.reduceToVectors(
    reducer=ee.Reducer.countEvery(),
    geometry=pilot_aoi,
    scale=10,
    geometryType="polygon",
    eightConnected=True,
    labelProperty="cluster_id",
    maxPixels=1e13,
    tileScale=4
)

def add_review_metrics(feature):
    area_ha = feature.geometry().area(maxError=1).divide(10000)
    centre = feature.geometry().centroid(maxError=1).coordinates()
    return feature.set({
        # Unique within this exported run; cluster_id is the mask label, not a unique region key.
        "region_id": feature.id(),
        "area_ha": area_ha,
        "longitude": centre.get(0),
        "latitude": centre.get(1)
    })

regions = (
    regions.map(add_review_metrics)
    .filter(ee.Filter.gte("area_ha", 0.1))
)

# Build evidence bands with distinct denominators: support is averaged
# only over observed radar pixels; coverage is averaged over all region pixels.
radar_coverage = vh_change.mask().unmask(0).rename("radar_coverage_share")
radar_support_band = radar_support.rename("radar_support_share")

evidence_image = ee.Image.cat([
    drop_2024.rename("ndvi_drop_2024"),
    drop_2025.rename("ndvi_drop_2025"),
    ndvi_change.rename("ndvi_change_2025_vs_2024"),
    radar_support_band,
    radar_coverage
])

# Measure each evidence band over the already-created polygons.
measured_regions = evidence_image.reduceRegions(
    collection=regions,
    reducer=ee.Reducer.mean(),
    scale=20,
    tileScale=4
)

ranked_regions = measured_regions.sort("area_ha", False)
total_regions = ranked_regions.size().getInfo()
top_features = ranked_regions.limit(20).getInfo().get("features", [])

print("--- Tree-cover-overlap review shortlist ---")
print("Connected regions >= 0.1 ha:", total_regions)
print("Largest regions (up to 20):")
print("Rank | Area ha | Centre lat, lon | NDVI drop 2024 | NDVI drop 2025 | NDVI change 2025 vs 2024 | Radar support share | Radar coverage share")

def fmt(value):
    return "NA" if value is None else f"{value:.3f}"

for rank, feature in enumerate(top_features, start=1):
    p = feature.get("properties", {})
    lat = p.get("latitude")
    lon = p.get("longitude")
    print(
        f"{rank:>2} | {p.get('area_ha', 0):.2f} | "
        f"{lat:.5f}, {lon:.5f} | "
        f"{fmt(p.get('ndvi_drop_2024'))} | "
        f"{fmt(p.get('ndvi_drop_2025'))} | "
        f"{fmt(p.get('ndvi_change_2025_vs_2024'))} | "
        f"{fmt(p.get('radar_support_share'))} | "
        f"{fmt(p.get('radar_coverage_share'))}"
    )

# Display outlines of the 20 largest regions.
shortlist = ee.FeatureCollection(top_features)
outline = ee.Image().byte().paint(shortlist, 1, 2).clip(pilot_aoi)
preview = outline.visualize(min=0, max=1, palette=["183b2a", "ffcc33"])
preview_url = preview.getThumbURL({
    "region": pilot_aoi,
    "dimensions": 800,
    "format": "png"
})
display(Image(url=preview_url))

print("\nRadar support share is the mean exploratory radar flag; coverage share is the fraction with radar observations.")
print("NA means the evidence band had no valid values in that region; it is not evidence of no disturbance.")
print("These are priority areas for human review, not confirmed deforestation or illegality.")


## 12. Unsupervised multi-sensor ML anomaly ranking

This section fits an Isolation Forest on a reproducible random reference sample of valid 2021 tree-cover pixels outside the persistent-candidate mask. The sample is split into model-fit and held-out calibration subsets. Seven optical/radar features are sampled across every connected candidate region that has usable pixels. Per-region anomaly percentiles and optical/radar feature medians are aggregated, all sampleable regions are ranked, and the top 20 model priorities proceed to the Hansen cross-check.

This is a real unsupervised machine-learning stage, but its output is a relative anomaly ranking—not a deforestation probability, supervised class, accuracy score, or finding of illegal activity. No human labels are used, and the model has not been validated against independent forest-loss labels.

Run Section 11 first in the same runtime.


In [ ]:
# Unsupervised multi-sensor anomaly model across all sampleable candidate regions.
import ee
from pathlib import Path
from urllib.request import urlopen

if "ranked_regions" not in globals():
    raise RuntimeError("Run Section 11 first in this Colab runtime.")
for required_name in [
    "pilot_aoi", "drop_2024", "drop_2025", "ndvi_change",
    "vv_change", "vh_change", "radar_support", "valid_all_years",
    "tree_cover_2021", "persistent_candidates", "total_regions",
]:
    if required_name not in globals():
        raise RuntimeError(
            "Section 12 needs the Section 6, 8, 10 and 11 layers. Missing: "
            + required_name
        )
if total_regions < 20:
    raise RuntimeError(
        f"At least 20 connected candidate regions are required; found {total_regions}."
    )

# Load the exact tested helper version from this repository commit.
module_path = Path("forestwatch_ml.py")
module_url = (
    "https://raw.githubusercontent.com/"
    "manjunathjadav360-rgb/ForestWatch-AI/98ffc5a78c7582c4d45f1262c4cd14db8e0cd57f/forestwatch_ml.py"
)
with urlopen(module_url, timeout=30) as response:
    module_source = response.read()
if len(module_source) < 1000:
    raise RuntimeError("Could not download the pinned ForestWatch ML module.")
module_path.write_bytes(module_source)

import importlib
import sys
sys.modules.pop("forestwatch_ml", None)
importlib.invalidate_caches()
from forestwatch_ml import (
    FEATURE_COLUMNS,
    fit_reference_model,
    score_region_samples,
)

# Seven features. Radar availability is explicit so missing observations are
# not silently treated as measured zero backscatter change.
radar_available = vh_change.mask().unmask(0).rename(
    "radar_observation_available"
)
ml_feature_image = ee.Image.cat([
    drop_2024.rename("ndvi_drop_2024"),
    drop_2025.rename("ndvi_drop_2025"),
    ndvi_change.rename("ndvi_change_2025_vs_2024"),
    vv_change.unmask(0).rename("vv_change_db"),
    vh_change.unmask(0).rename("vh_change_db"),
    radar_support.unmask(0).rename("radar_support_flag"),
    radar_available,
]).updateMask(valid_all_years).clip(pilot_aoi)

# The reference distribution comes from valid 2021 tree-cover pixels outside
# the persistent-candidate mask, not from the candidate pixels being ranked.
non_candidate_tree_cover = (
    tree_cover_2021.eq(1)
    .And(persistent_candidates.unmask(0).eq(0))
)
ml_reference_image = ml_feature_image.updateMask(non_candidate_tree_cover)

reference_fc = ml_reference_image.sample(
    region=pilot_aoi,
    scale=20,
    numPixels=4000,
    seed=42,
    dropNulls=True,
    geometries=False,
    tileScale=4,
)
reference_records = reference_fc.getInfo().get("features", [])
print("Valid non-candidate tree-cover reference pixels:", len(reference_records))
if len(reference_records) < 100:
    raise RuntimeError(
        "Too few non-candidate tree-cover reference pixels for unsupervised ML. "
        "Check WorldCover coverage and optical/radar masks."
    )

model_bundle = fit_reference_model(
    reference_records,
    feature_names=FEATURE_COLUMNS,
    contamination=0.05,
    random_state=42,
)

# Give every Section 11 polygon a stable area rank before candidate sampling.
area_sorted_list = ranked_regions.sort("area_ha", False).toList(total_regions)

def add_area_rank(rank):
    feature = ee.Feature(area_sorted_list.get(ee.Number(rank).subtract(1)))
    return feature.set("area_rank", rank)

area_ranked_regions = ee.FeatureCollection(
    ee.List.sequence(1, total_regions).map(add_area_rank)
)

# Sample across all connected regions, not only the 20 largest.
# Earth Engine's interactive getInfo() response is limited to 5,000
# elements. Sample a small, reproducible number of valid pixels PER region,
# then merge on the server before retrieving the bounded result.
# Four samples per region keeps the 790-region pilot below 3,200 pixels.
if total_regions > 1000:
    raise RuntimeError(
        "This pilot has more than 1,000 regions. Increase the sampling "
        "strategy in bounded batches before running Section 12."
    )

def sample_one_region(feature, accumulated):
    feature = ee.Feature(feature)
    region_samples = ml_feature_image.sample(
        region=feature.geometry(),
        scale=20,
        numPixels=4,
        seed=42,
        dropNulls=True,
        geometries=False,
        tileScale=4,
    ).map(
        lambda pixel: ee.Feature(pixel).set({
            "region_id": feature.get("region_id"),
            "area_rank": feature.get("area_rank"),
        })
    )
    return ee.FeatureCollection(accumulated).merge(region_samples)

candidate_fc = ee.FeatureCollection(
    area_ranked_regions.iterate(
        sample_one_region,
        ee.FeatureCollection([]),
    )
)
candidate_records = candidate_fc.getInfo().get("features", [])
if not candidate_records:
    raise RuntimeError(
        "No candidate pixels could be sampled. Check the overlap of the "
        "candidate polygons with the valid optical/radar feature image."
    )

region_scores = score_region_samples(model_bundle, candidate_records)
if len(region_scores) < 20:
    raise RuntimeError(
        "Fewer than 20 candidate regions had valid ML samples: "
        + str(len(region_scores))
    )

# Join pixel-level anomaly aggregates back to region polygons on the server,
# preserving geometry for the final top-20 export.
score_features = [
    ee.Feature(None, {"region_id": region_id, **metrics})
    for region_id, metrics in region_scores.items()
]
score_collection = ee.FeatureCollection(score_features)
joined_regions = ee.FeatureCollection(
    ee.Join.saveFirst("ml_score").apply(
        primary=area_ranked_regions,
        secondary=score_collection,
        condition=ee.Filter.equals(
            leftField="region_id",
            rightField="region_id",
        ),
    )
).filter(ee.Filter.notNull(["ml_score"]))

def attach_ml_metrics(feature):
    score_feature = ee.Feature(feature.get("ml_score"))
    # Keep the anomaly score and its interpretable optical/radar evidence
    # profile together on the candidate polygon.
    metric_names = [
        "ml_anomaly_percentile_median",
        "ml_anomaly_p95_share",
        "ml_sample_count",
        "ndvi_drop_2024_median",
        "ndvi_drop_2025_median",
        "ndvi_change_2025_vs_2024_median",
        "vv_change_db_median",
        "vh_change_db_median",
        "ml_radar_observation_share",
        "ml_radar_support_observed_share",
    ]
    # Dictionary transfer preserves genuinely missing radar metrics as
    # absent fields instead of manufacturing null/zero measurements.
    return ee.Feature(feature).set(
        score_feature.toDictionary(metric_names)
    )

ml_ranked_all = (
    joined_regions
    .map(attach_ml_metrics)
    .sort("ml_anomaly_percentile_median", False)
)
ml_scored_region_count = ml_ranked_all.size().getInfo()
if ml_scored_region_count < 20:
    raise RuntimeError(
        "Fewer than 20 regions remained after matching ML scores to geometries: "
        + str(ml_scored_region_count)
    )

ml_top20_features = ml_ranked_all.limit(20).getInfo().get("features", [])
if len(ml_top20_features) != 20:
    raise RuntimeError(
        "Could not retrieve all 20 ML-ranked geometries; export was stopped."
    )

ml_review_regions = ee.FeatureCollection([
    ee.Feature(feature_json).set("ml_priority_rank", priority_rank)
    for priority_rank, feature_json in enumerate(ml_top20_features, start=1)
])

print("\n--- ForestWatch unsupervised ML results ---")
print("Algorithm: StandardScaler + IsolationForest")
print("Reference pixels returned:", model_bundle["reference_sample_count"])
print("Model-fit pixels:", model_bundle["training_sample_count"])
print("Held-out calibration pixels:", model_bundle["calibration_sample_count"])
print("Candidate pixels scored across all regions:", len(candidate_records))
print("Candidate regions with valid ML samples:", ml_scored_region_count)
print("Candidate regions without a valid ML sample:", total_regions - ml_scored_region_count)
print("Input features:", ", ".join(FEATURE_COLUMNS))
print("ML rank | Original area rank | Area ha | Median anomaly percentile | Pixels scored | Share >= 95th percentile")

for priority_rank, feature_json in enumerate(ml_top20_features, start=1):
    properties = feature_json.get("properties", {})
    print(
        f"{priority_rank:>2} | "
        f"{int(properties.get('area_rank', 0)):>3} | "
        f"{float(properties.get('area_ha', 0) or 0):.2f} | "
        f"{float(properties.get('ml_anomaly_percentile_median', 0)):.1f} | "
        f"{int(properties.get('ml_sample_count', 0))} | "
        f"{float(properties.get('ml_anomaly_p95_share', 0)) * 100:.1f}%"
    )

print("\nInterpretation:")
print("- Higher percentiles mean more unusual feature patterns relative to the held-out non-candidate tree-cover calibration sample.")
print("- Percentiles are relative anomaly ranks, not probabilities or accuracy measurements.")
print("- The model is unsupervised and has not been validated against independent forest-loss labels.")
print("- All regions with valid candidate-pixel samples are ranked; the top 20 anomaly priorities proceed to Hansen cross-check.")


## 13. Independent forest-loss cross-check (Hansen Global Forest Change)

This section cross-checks the 20 ML-priority regions selected by Section 12 against Hansen/UMD Global Forest Change v1.13 (approximately 30 m pixels). It reports mean historical year-2000 canopy cover, the fraction of sampled pixels flagged for loss in 2021–2025, annual loss shares, and the approximate mapped area of Hansen loss pixels intersecting each candidate polygon. This is a separate evidence source, not model training data or ground truth. Results remain approximate because the Hansen grid is coarser than the 10 m candidate polygons; they are not proof of illegality.

Run Sections 11 and 12 first in the same runtime.


In [ ]:
# Independent cross-check using Hansen Global Forest Change v1.13.
import ee

if "ml_review_regions" not in globals():
    raise RuntimeError("Run Section 12 first in this Colab runtime.")

print("--- Hansen Global Forest Change independent cross-check ---")

gfc = ee.Image("UMD/hansen/global_forest_change_2025_v1_13")
treecover2000 = gfc.select("treecover2000").rename("treecover2000_pct")
lossyear = gfc.select("lossyear")
pixel_area_ha = ee.Image.pixelArea().divide(10000)

share_bands = [treecover2000]
area_bands = []

# Hansen loss-year codes: 21=2021, ..., 25=2025.
for year in range(2021, 2026):
    loss_flag = lossyear.eq(year - 2000)
    share_bands.append(
        loss_flag.unmask(0).rename("loss_" + str(year) + "_share")
    )
    area_bands.append(
        pixel_area_ha.updateMask(loss_flag).rename(
            "loss_area_" + str(year) + "_ha"
        )
    )

loss_2021_2025 = lossyear.gte(21).And(lossyear.lte(25))
share_bands.append(
    loss_2021_2025.unmask(0).rename("loss_2021_2025_share")
)
area_bands.append(
    pixel_area_ha.updateMask(loss_2021_2025).rename(
        "loss_area_2021_2025_ha"
    )
)

# Mean reducers produce pixel fractions; sum reducers measure mapped
# Hansen loss-pixel area intersecting each candidate polygon.
gfc_evidence = ee.Image.cat(share_bands + area_bands)
combined_reducer = ee.Reducer.mean().combine(
    reducer2=ee.Reducer.sum(),
    sharedInputs=True
)

gfc_review = ml_review_regions
gfc_review_with_stats = gfc_evidence.reduceRegions(
    collection=gfc_review,
    reducer=combined_reducer,
    scale=30,
    tileScale=4
)

gfc_results = gfc_review_with_stats.getInfo()
features = gfc_results.get("features", [])

print(
    "ML rank | Candidate ha | Tree cover 2000 mean % | "
    "Hansen loss share 2021–25 % | Hansen loss-pixel area ha"
)

valid_regions = 0
regions_with_loss_signal = 0
total_candidate_area_ha = 0.0
total_hansen_loss_pixel_area_ha = 0.0
annual_loss_pixel_area_ha = {year: 0.0 for year in range(2021, 2026)}

def fmt_pct(value):
    return "NA" if value is None else f"{value * 100:.2f}"

def fmt_area(value):
    return "NA" if value is None else f"{value:.4f}"

for rank, feature in enumerate(features, start=1):
    props = feature.get("properties", {})
    candidate_area = props.get("area_ha") or 0.0
    treecover_mean = props.get("treecover2000_pct_mean")
    loss_share = props.get("loss_2021_2025_share_mean")
    loss_area = props.get("loss_area_2021_2025_ha_sum")

    if loss_share is not None and candidate_area > 0:
        valid_regions += 1
        total_candidate_area_ha += candidate_area
        if loss_share > 0:
            regions_with_loss_signal += 1

    if loss_area is not None:
        total_hansen_loss_pixel_area_ha += loss_area

    print(
        f"{rank:>2} | {candidate_area:.2f} | "
        f"{fmt_pct(treecover_mean)} | "
        f"{fmt_pct(loss_share)} | "
        f"{fmt_area(loss_area)}"
    )

    for year in range(2021, 2026):
        annual_area = props.get(
            "loss_area_" + str(year) + "_ha_sum"
        )
        if annual_area is not None:
            annual_loss_pixel_area_ha[year] += annual_area

print("\n--- Hansen cross-check summary (not an accuracy evaluation) ---")
print("Top-ranked regions returned:", len(features))
print("Regions with valid Hansen loss-share data:", valid_regions)
print(
    "Regions with at least one sampled Hansen loss pixel:",
    f"{regions_with_loss_signal}/{valid_regions}"
)
print(
    "Total candidate polygon area evaluated (ha):",
    f"{total_candidate_area_ha:.2f}"
)
print(
    "Approximate Hansen loss-pixel area intersecting those polygons (ha):",
    f"{total_hansen_loss_pixel_area_ha:.4f}"
)

if total_candidate_area_ha > 0:
    print(
        "Approximate Hansen loss-pixel area / candidate polygon area (%):",
        f"{100 * total_hansen_loss_pixel_area_ha / total_candidate_area_ha:.4f}"
    )

for year in range(2021, 2026):
    print(
        f"Approximate Hansen loss-pixel area intersecting candidates in {year} (ha):",
        f"{annual_loss_pixel_area_ha[year]:.4f}"
    )

print("\nInterpretation:")
print("- Hansen loss is an independent cross-check signal, not ground-truth confirmation.")
print("- Pixel-area sums are approximate intersections with candidate polygons at about 30 m resolution.")
print("- These hectares are mapped Hansen loss-pixel area, not confirmed deforested area.")
print("- Hansen treecover2000 is historical canopy cover, not a 2021 forest map.")
print("- No Hansen loss flag does not rule out recent, small, or undetected change.")
print("- Hansen loss can reflect harvesting, fire, storms, disease, or other stand-replacement disturbance.")
print("- These results do not establish natural-forest loss, cause, or illegality.")


## 14. Export the top 20 ML-priority regions with Hansen cross-check fields

Exports the 20 highest anomaly-ranked candidate polygons from Section 12, retaining each region's original Section 11 area rank and adding Section 13 Hansen summary fields. CSV and GeoJSON exports are saved to the `ForestWatchAI` folder in Google Drive. Run Sections 11–13 first in this same runtime.


In [ ]:
# Export the top 20 ranked candidate regions with Hansen cross-check properties.

import ee

if "gfc_review_with_stats" not in globals():
    raise RuntimeError("Run Section 13 first in this Colab runtime.")

export_regions = gfc_review_with_stats
export_fields = [
    "region_id", "ml_priority_rank", "area_rank", "area_ha", "latitude", "longitude",
    "ndvi_drop_2024", "ndvi_drop_2025",
    "ndvi_change_2025_vs_2024",
    "radar_support_share", "radar_coverage_share",
    "ml_anomaly_percentile_median", "ml_anomaly_p95_share", "ml_sample_count",
    "ndvi_drop_2024_median", "ndvi_drop_2025_median",
    "ndvi_change_2025_vs_2024_median",
    "vv_change_db_median", "vh_change_db_median",
    "ml_radar_observation_share", "ml_radar_support_observed_share",
    "treecover2000_pct_mean",
    "loss_2021_share_mean", "loss_2022_share_mean",
    "loss_2023_share_mean", "loss_2024_share_mean",
    "loss_2025_share_mean", "loss_2021_2025_share_mean",
    "loss_area_2021_ha_sum", "loss_area_2022_ha_sum",
    "loss_area_2023_ha_sum", "loss_area_2024_ha_sum",
    "loss_area_2025_ha_sum", "loss_area_2021_2025_ha_sum"
]

csv_task = ee.batch.Export.table.toDrive(
    collection=export_regions,
    description="ForestWatch_Top20_Hansen_CSV",
    folder="ForestWatchAI",
    fileNamePrefix="forestwatch_top20_candidates_with_hansen",
    fileFormat="CSV",
    selectors=export_fields
)

geojson_task = ee.batch.Export.table.toDrive(
    collection=export_regions,
    description="ForestWatch_Top20_Hansen_GeoJSON",
    folder="ForestWatchAI",
    fileNamePrefix="forestwatch_top20_candidates_with_hansen",
    fileFormat="GeoJSON"
)

csv_task.start()
geojson_task.start()

print("CSV export task:", csv_task.id)
print("GeoJSON export task:", geojson_task.id)
print("Open the Earth Engine Tasks panel and confirm both tasks complete.")
print("Expected Drive folder: My Drive / ForestWatchAI")
